# درس 4: قانون اعداد بزرگ و قضیه حد مرکزی

متن کامل درس: [https://quantacademy.ir/courses/quant-math/law-of-large-numbers-clt/](https://quantacademy.ir/courses/quant-math/law-of-large-numbers-clt/)

داده‌های این دوره **شبیه‌سازی‌شده** هستند، نه قیمت واقعی بازار؛ هدف یاد دادن روش است. آموزشی است و توصیه سرمایه‌گذاری نیست.

سلول‌ها را از بالا به پایین اجرا کن (Shift+Enter).

## ۱. قانون اعداد بزرگ

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

rng = np.random.default_rng(11)
p, reps, sigma = 0.55, 2000, 9.95
print(f"{'n':>7} {'theory_SE':>10} {'sim_std':>9} {'mean_of_means':>14}")
for n in (10, 100, 1_000, 10_000, 100_000):
    k = rng.binomial(n, p, size=reps)
    means = 10 * (2 * k / n - 1)
    print(f"{n:>7} {sigma / np.sqrt(n):>10.3f} {means.std(ddof=1):>9.3f} {means.mean():>14.3f}")

## ۲. قضیه حد مرکزی

In [ ]:
rng = np.random.default_rng(12)
reps = 20000
print(f"{'n':>4} {'skew_sim':>9} {'skew_theory':>12} {'below_-1.96':>12} {'above_+1.96':>12}")
for n in (1, 2, 5, 30, 200):
    m = rng.exponential(1.0, size=(reps, n)).mean(axis=1)
    zs = (m - 1.0) / (1.0 / np.sqrt(n))
    print(f"{n:>4} {stats.skew(m):>9.2f} {2 / np.sqrt(n):>12.2f} {(zs < -1.96).mean():>12.3f} {(zs > 1.96).mean():>12.3f}")

## ۳. وقتی CLT می‌شکند: دم پهن

In [ ]:
rng = np.random.default_rng(13)
reps = 4000
def iqr(a):
    return np.subtract(*np.percentile(a, [75, 25]))

print(f"{'n':>6} {'normal':>8} {'t(3)':>8} {'cauchy':>8}")
for n in (1, 10, 100, 1_000):
    row = [
        iqr(rng.standard_normal((reps, n)).mean(axis=1)),
        iqr(rng.standard_t(3, (reps, n)).mean(axis=1)),
        iqr(rng.standard_cauchy((reps, n)).mean(axis=1)),
    ]
    print(f"{n:>6} {row[0]:>8.3f} {row[1]:>8.3f} {row[2]:>8.3f}")

## ۴. خطای استاندارد بازده روزانه

In [ ]:
URL = "https://raw.githubusercontent.com/quantacademyir/quant-math/main/data/btc_daily_5y.csv"
close = pd.read_csv(URL, parse_dates=["date"], index_col="date")["close"]
r = close.pct_change().dropna()
n, m, s = len(r), r.mean(), r.std()
se = s / np.sqrt(n)
print(n, f"{m:.5f}", f"{s:.4f}", f"{se:.5f}", round(m / se, 2))
print(f"{m - 1.96 * se:.5f} {m + 1.96 * se:.5f}")
need_days = (2 * s / m) ** 2
print(round(need_days), round(need_days / 365, 1))

## ۵. Sharpe و سال‌های لازم

In [ ]:
rng = np.random.default_rng(14)
SR, sd_d, reps = 0.5, 0.02, 200_000
mu_d = SR * sd_d / np.sqrt(365)
print(f"{'years':>6} {'SE_sim':>8} {'SE_formula':>11} {'P(est<0)':>9}")
for years in (1, 3, 5, 10, 20):
    n = 365 * years
    mean_hat = rng.normal(mu_d, sd_d / np.sqrt(n), size=reps)
    var_hat = sd_d**2 * rng.chisquare(n - 1, size=reps) / (n - 1)
    sr_hat = mean_hat / np.sqrt(var_hat) * np.sqrt(365)
    print(f"{years:>6} {sr_hat.std():>8.3f} {np.sqrt((1 + SR**2 / (2 * 365)) / years):>11.3f} {(sr_hat < 0).mean():>9.1%}")